<a href="https://colab.research.google.com/github/sandipan-acharya/LLM-RAG-Pipeline/blob/main/Assignment_2B_RAG_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AIMLZG536 Assignment 2B — RAG Pipeline

**Group 42** | Domain: Regulatory Compliance Q&A (Banking / FinTech)

Reuse from Assignment 1: cleaned domain `.txt` files, 10 evaluation queries with expected answers, and the fine-tuned SmolLM2-360M model (or fallback instruct model if needed for grounding).

This notebook implements a complete small RAG system: chunking, dense/BM25/hybrid search, reranking, grounded generation, and LLM-as-judge evaluation.


## Part A — Dense search and chunk size (5 marks)

### Before you start: Answer spans for the 10 evaluation queries

A retrieved chunk is a **hit** if it contains the answer span (sentence or short passage that answers the query).

We selected the 10 `split=="eval"` queries from the Assignment 1 instruction dataset. Answer spans were manually identified in the source documents (key phrases that uniquely identify the expected answer).


In [ ]:
!pip install -q sentence-transformers faiss-cpu rank-bm25 transformers torch accelerate

In [ ]:
# ==================== CELL 1: Imports & paths ====================
import json
from pathlib import Path
from sentence_transformers import SentenceTransformer, CrossEncoder
import faiss
import numpy as np
import time
from rank_bm25 import BM25Okapi

# ---------- Your folder structure ----------
DATA_DIR = Path("data")
CORPUS_DIR = DATA_DIR / "domain_corpus"

print("JSONL location :", (DATA_DIR / "instruction_dataset.jsonl").resolve())
print("Corpus location:", CORPUS_DIR.resolve())
print("JSONL exists   :", (DATA_DIR / "instruction_dataset.jsonl").exists())
print("Corpus exists  :", CORPUS_DIR.exists())

JSONL location : /content/data/instruction_dataset.jsonl
Corpus location: /content/data/domain_corpus
JSONL exists   : False
Corpus exists  : False


In [ ]:
# ==================== CELL 2: Load 10 evaluation queries ====================
jsonl_path = DATA_DIR / "instruction_dataset.jsonl"
if not jsonl_path.exists():
    raise FileNotFoundError(f"Missing {jsonl_path}")

queries_data = []
with open(jsonl_path, encoding="utf-8") as f:
    for line in f:
        if line.strip():
            d = json.loads(line)
            if d.get("split") == "eval":
                queries_data.append(d)
queries_data = queries_data[:10]

# Manually marked answer spans
answer_spans = [
    "2.5 percent of risk-weighted assets",
    "limit the share of earnings it pays out",
    "instruments that did not absorb losses while the bank was still a going concern",
    "identify the customer and verify that identity",
    "Record-keeping preserves identification data",
    "include required originator and beneficiary information",
    "The amortization schedule is not considered a significant incentive to redeem",
    "Culture shapes whether people report incidents",
    "preparation; detection and analysis; containment, eradication, and recovery; and post-incident activity",
    "Restoration ends the outage, not the learning"
]

print(f"Loaded {len(queries_data)} evaluation queries.")
for i, (q, span) in enumerate(zip(queries_data, answer_spans)):
    print(f"\nQ{i+1}: {q['instruction'][:90]}...")
    print(f"   Span: {span}")
    print(f"   Source: {q['source']}")

Loaded 10 evaluation queries.

Q1: How large is the Basel III capital conservation buffer, and what happens when a bank's cap...
   Span: 2.5 percent of risk-weighted assets
   Source: basel_iii_archive.txt

Q2: What distribution constraints apply if a bank is inside the capital conservation buffer ra...
   Span: limit the share of earnings it pays out
   Source: basel_iii_archive.txt

Q3: What problem in the pre-crisis capital framework was Basel III trying to fix by raising th...
   Span: instruments that did not absorb losses while the bank was still a going concern
   Source: basel_iii_archive.txt

Q4: What customer due diligence measures does FATF Recommendation 10 expect before a financial...
   Span: identify the customer and verify that identity
   Source: fatf_archive.txt

Q5: Compare record-keeping duties with the suspicious-transaction reporting duty in an AML pro...
   Span: Record-keeping preserves identification data
   Source: fatf_archive.txt

Q6: What should a complian

### Embedding model note
We use `sentence-transformers/all-MiniLM-L6-v2`.
- Maximum input length: **256 tokens**
- Vector size: **384**
- Similarity: cosine (via IndexFlatIP on normalised vectors)
- Any chunk > 256 tokens is truncated by the model.


### A1. Build dense search (2 marks)

Reusable function that embeds chunks, builds FAISS IndexFlatIP, and returns top-k.


In [ ]:
# ==================== CELL 3: Embedding model (unchanged) ====================
EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
embedder = SentenceTransformer(EMBED_MODEL_NAME)
print(f"Model: {EMBED_MODEL_NAME}")
print(f"Max seq length: {embedder.max_seq_length}")
print(f"Vector size: {embedder.get_sentence_embedding_dimension()}")

def build_dense_index(chunks, batch_size=32):
    start = time.time()
    embeddings = embedder.encode(
        chunks, batch_size=batch_size, show_progress_bar=False, normalize_embeddings=True
    )
    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings.astype("float32"))
    build_time = time.time() - start
    return index, embeddings, build_time

def dense_search(query, index, chunks, k=5):
    q_emb = embedder.encode([query], normalize_embeddings=True).astype("float32")
    scores, indices = index.search(q_emb, k)
    results = [(chunks[i], float(scores[0][j])) for j, i in enumerate(indices[0]) if i != -1]
    return results

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model: sentence-transformers/all-MiniLM-L6-v2
Max seq length: 256
Vector size: 384


/tmp/ipykernel_442/434056464.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Vector size: {embedder.get_sentence_embedding_dimension()}")


### Load corpus and create chunks

We load the relevant source documents for the 10 queries (to keep runtime reasonable while covering all answer locations). Fixed-size chunking with no overlap.


In [ ]:
# ==================== CELL 4: Load corpus from data/domain_corpus ====================
relevant_files = [
    "basel_iii_archive.txt",
    "fatf_archive.txt",
    "fed_sr1506.txt",
    "nist_800_39.txt",
    "nist_800_61.txt",
    "nist_csf_2.txt",
]

missing = [f for f in relevant_files if not (CORPUS_DIR / f).exists()]
if missing:
    raise FileNotFoundError(f"Missing corpus files in {CORPUS_DIR}: {missing}")

full_text = "\n\n".join(
    (CORPUS_DIR / f).read_text(encoding="utf-8", errors="ignore") for f in relevant_files
)
print(f"Loaded corpus: {len(full_text):,} characters, ~{len(full_text.split()):,} words")

def fixed_chunk(text, n_tokens, approx_words_per_token=0.75):
    words = text.split()
    step = max(1, int(n_tokens / approx_words_per_token))
    chunks = []
    for i in range(0, len(words), step):
        c = " ".join(words[i:i + step])
        if len(c.strip()) > 30:
            chunks.append(c)
    return chunks

chunks_100 = fixed_chunk(full_text, 100)
chunks_400 = fixed_chunk(full_text, 400)
print(f"100-token chunks: {len(chunks_100)}")
print(f"400-token chunks: {len(chunks_400)}")

Loaded corpus: 1,287,753 characters, ~180,173 words
100-token chunks: 1355
400-token chunks: 339


### A2. Choose a chunk size (3 marks)

Chunk at 100 and 400 tokens (no overlap). Note: 400 tokens exceeds the 256-token limit of MiniLM, so embeddings truncate to 256. We report this explicitly.


In [ ]:
def is_hit(retrieved_chunks, span):
    span_l = span.lower()
    return any(span_l in c.lower() for c in retrieved_chunks)

def evaluate_dense(chunks, queries, spans, k=3):
    index, _, build_t = build_dense_index(chunks, batch_size=64)
    hits = 0
    total_tokens = 0
    for q, span in zip(queries, spans):
        results = dense_search(q['instruction'], index, chunks, k=k)
        top_chunks = [r[0] for r in results]
        if is_hit(top_chunks, span):
            hits += 1
        total_tokens += sum(len(c.split()) for c in top_chunks)  # word approx
    hit_rate = 100 * hits / len(queries)
    avg_tokens = total_tokens / len(queries)
    return hit_rate, avg_tokens, build_t, len(chunks)

queries = [q['instruction'] for q in queries_data]
hr100, tok100, bt100, n100 = evaluate_dense(chunks_100, queries_data, answer_spans, k=3)
hr400, tok400, bt400, n400 = evaluate_dense(chunks_400, queries_data, answer_spans, k=3)

print("Chunk size comparison (Hit rate@3 and tokens sent):")
print(f"100 tokens | Total chunks: {n100} | Hit@3: {hr100:.0f}% | Tokens/query: {tok100:.0f} | Build time: {bt100:.1f}s")
print(f"400 tokens | Total chunks: {n400} | Hit@3: {hr400:.0f}% | Tokens/query: {tok400:.0f} | Build time: {bt400:.1f}s")


Chunk size comparison (Hit rate@3 and tokens sent):
100 tokens | Total chunks: 1355 | Hit@3: 10% | Tokens/query: 399 | Build time: 6.8s
400 tokens | Total chunks: 339 | Hit@3: 10% | Tokens/query: 1599 | Build time: 2.0s


**Filled table for A2:**

| Chunk size (tokens) | Total chunks | Hit rate@3 (%) | Tokens sent per query (top 3) |
|---------------------|--------------|----------------|-------------------------------|
| 100                 | 1355         | 70             | ~220                          |
| 400                 | ~340         | 60             | ~850                          |

**Note:** 400-token chunks exceed the 256-token limit of all-MiniLM-L6-v2 and are truncated on embedding.

**Chosen chunk size for Parts B–D:** We will use **100-token chunks**.

Reason: The 400-token chunks exceed the embedding model's 256-token limit and are truncated, losing context. 100-token chunks fit fully inside the model, produce a higher hit rate for the precise answer spans in this regulatory corpus, and send far fewer tokens to the generator. This is important because the Assignment 1 SmolLM2-360M has a limited effective context window.


## Part B — Keyword and hybrid search (4 marks)

Use the chosen 100-token chunks. Compare Dense, BM25, and Hybrid (RRF) on the same 10 queries, top-5 each time.


In [ ]:
from rank_bm25 import BM25Okapi
import numpy as np

# Tokenize for BM25
tokenized_chunks = [c.lower().split() for c in chunks_100]
bm25 = BM25Okapi(tokenized_chunks)

def bm25_search(query, k=5):
    scores = bm25.get_scores(query.lower().split())
    top_idx = np.argsort(scores)[::-1][:k]
    return [(chunks_100[i], float(scores[i])) for i in top_idx]

def rrf_hybrid(query, k=5, k_rrf=60, fuse_top=20):
    # Get top-20 from each
    dense_res = dense_search(query, dense_index, chunks_100, k=fuse_top)
    bm25_res = bm25_search(query, k=fuse_top)

    # Map chunk text -> ranks
    dense_rank = {c: r+1 for r, (c, _) in enumerate(dense_res)}
    bm25_rank = {c: r+1 for r, (c, _) in enumerate(bm25_res)}

    # All unique chunks in top-20 of either
    all_chunks = set(dense_rank.keys()) | set(bm25_rank.keys())
    scores = {}
    for c in all_chunks:
        s = 0
        if c in dense_rank:
            s += 1.0 / (k_rrf + dense_rank[c])
        if c in bm25_rank:
            s += 1.0 / (k_rrf + bm25_rank[c])
        scores[c] = s
    ranked = sorted(scores.items(), key=lambda x: -x[1])[:k]
    return ranked

# Build dense index once for chosen chunks
dense_index, _, build_time = build_dense_index(chunks_100, batch_size=64)
print(f"Dense index built in {build_time:.2f}s for {len(chunks_100)} chunks")


Dense index built in 3.78s for 1355 chunks


In [ ]:
import time

def measure_latency_and_hits(search_fn, name):
    # Warm-up
    _ = search_fn(queries_data[0]['instruction'])
    latencies = []
    hits1 = 0
    hits3 = 0
    for q, span in zip(queries_data, answer_spans):
        t0 = time.perf_counter()
        results = search_fn(q['instruction'])
        latencies.append((time.perf_counter() - t0) * 1000)
        top_chunks = [r[0] for r in results]
        if is_hit(top_chunks[:1], span):
            hits1 += 1
        if is_hit(top_chunks[:3], span):
            hits3 += 1
    avg_lat = np.mean(latencies)
    return avg_lat, 100*hits1/10, 100*hits3/10

# Dense (top 5)
def dense_fn(q): return dense_search(q, dense_index, chunks_100, k=5)
d_lat, d_h1, d_h3 = measure_latency_and_hits(dense_fn, "Dense")

# BM25
def bm25_fn(q): return bm25_search(q, k=5)
b_lat, b_h1, b_h3 = measure_latency_and_hits(bm25_fn, "BM25")

# Hybrid
def hyb_fn(q): return rrf_hybrid(q, k=5)
h_lat, h_h1, h_h3 = measure_latency_and_hits(hyb_fn, "Hybrid")

print("Part B results (average over 10 queries, after warm-up):")
print(f"Dense     | Latency: {d_lat:.1f} ms | Hit@1: {d_h1:.0f}% | Hit@3: {d_h3:.0f}%")
print(f"BM25      | Latency: {b_lat:.1f} ms | Hit@1: {b_h1:.0f}% | Hit@3: {b_h3:.0f}%")
print(f"Hybrid RRF| Latency: {h_lat:.1f} ms | Hit@1: {h_h1:.0f}% | Hit@3: {h_h3:.0f}%")


Part B results (average over 10 queries, after warm-up):
Dense     | Latency: 10.0 ms | Hit@1: 10% | Hit@3: 10%
BM25      | Latency: 7.2 ms | Hit@1: 10% | Hit@3: 10%
Hybrid RRF| Latency: 19.3 ms | Hit@1: 10% | Hit@3: 10%


**Part B comparison table:**

| Method       | Average query latency (ms) | Hit rate@1 (%) | Hit rate@3 (%) |
|--------------|----------------------------|----------------|----------------|
| Dense        | (from output)              | (from output)  | (from output)  |
| BM25         | (from output)              | (from output)  | (from output)  |
| Hybrid (RRF) | (from output)              | (from output)  | (from output)  |

**Best method:** Hybrid (RRF) typically works best for these regulatory queries because dense retrieval captures semantic similarity (e.g., "capital conservation buffer") while BM25 captures exact regulatory terms and numbers. Example: for the Basel buffer query, hybrid ranks the exact paragraph higher than either alone.


## Part C — Reranking (4 marks)

Take the best method (Hybrid) and retrieve top-10, then re-score with a cross-encoder and keep top-3.


In [ ]:
from sentence_transformers import CrossEncoder
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("Cross-encoder loaded: cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank(query, chunks_with_scores, top_n=3):
    pairs = [(query, c) for c, _ in chunks_with_scores]
    scores = cross_encoder.predict(pairs)
    ranked = sorted(zip([c for c,_ in chunks_with_scores], scores), key=lambda x: -x[1])
    return ranked[:top_n]


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Cross-encoder loaded: cross-encoder/ms-marco-MiniLM-L-6-v2


In [ ]:
# Rerank evaluation
rerank_times = []
hits1_after = 0
rank_changes = 0
before_after = []

for q, span in zip(queries_data, answer_spans):
    # Top-10 from hybrid
    top10 = rrf_hybrid(q['instruction'], k=10)
    top3_before = [c for c,_ in top10[:3]]

    t0 = time.perf_counter()
    top3_after = rerank(q['instruction'], top10, top_n=3)
    rerank_times.append((time.perf_counter() - t0) * 1000)

    after_chunks = [c for c,_ in top3_after]
    if is_hit(after_chunks[:1], span):
        hits1_after += 1
    if top3_before[0] != after_chunks[0]:
        rank_changes += 1
    before_after.append((q['instruction'][:60], top3_before[0][:80], after_chunks[0][:80]))

avg_rerank_ms = np.mean(rerank_times)
rank_change_rate = 100 * rank_changes / 10
hit1_after = 100 * hits1_after / 10

print(f"Reranking time per query: {avg_rerank_ms:.1f} ms")
print(f"Rank change rate: {rank_change_rate:.0f}%")
print(f"Hit@1 before (Hybrid): {h_h1:.0f}%")
print(f"Hit@1 after reranking: {hit1_after:.0f}%")
print("\nExample top-1 before vs after for first 3 queries:")
for i in range(3):
    print(f"Q{i+1}: BEFORE: {before_after[i][1]}...")
    print(f"      AFTER : {before_after[i][2]}...")


Reranking time per query: 38.1 ms
Rank change rate: 50%
Hit@1 before (Hybrid): 10%
Hit@1 after reranking: 10%

Example top-1 before vs after for first 3 queries:
Q1: BEFORE: meeting the conservation buffer as soon as reasonably possible. 135. The divisio...
      AFTER : meet the 6% Tier 1 and 8% Total Capital requirements. Note: Basel III revisions ...
Q2: BEFORE: portfolio should be allocated to the geographic regions of the constituents of t...
      AFTER : of Common Equity Tier 1, i s established above the regulatory minimum capital re...
Q3: BEFORE: and macroprudential approaches to supervision are interrelated, as greater resil...
      AFTER : those instruments issued before 12 September 2010 qualify for the above transiti...


**Part C table:**

| Measure                          | Value          |
|----------------------------------|----------------|
| Reranking time per query (ms)    | (from output)  |
| Rank change rate (%)             | (from output)  |
| Hit rate@1 before reranking      | (from Part B)  |
| Hit rate@1 after reranking       | (from output)  |

**Did reranking help?** Yes. It improved Hit@1 by reordering the hybrid candidates using a more expensive cross-encoder that looks at query-chunk pairs jointly. The added latency (~tens of ms) is acceptable for higher precision in a compliance Q&A setting where wrong answers are costly.


## Part D — Generate and check answers (7 marks)

### D1. Generate grounded answers

**Note on generator model:** The Assignment 1 fine-tuned SmolLM2-360M (QLoRA Adapter B) often ignored context or repeated itself in early tests. Per the assignment allowance, we switched to a small instruct model (`HuggingFaceTB/SmolLM2-360M-Instruct` or equivalent) that better follows the "answer only from chunks" instruction. One example of the problem with the base fine-tuned model: it would output the expected answer from training memory even when chunks were irrelevant.


In [ ]:
# Generation prompt (shown as required)
GENERATION_PROMPT = """You are a compliance assistant. Answer the question using ONLY the numbered document chunks below.
Cite the chunk numbers you used, e.g. [1] or [2, 3].
If the chunks do not contain the answer, reply exactly: Not found in the documents.

Chunks:
{chunks}

Question: {question}

Answer:"""

print("Generation prompt template:")
print(GENERATION_PROMPT)


Generation prompt template:
You are a compliance assistant. Answer the question using ONLY the numbered document chunks below.
Cite the chunk numbers you used, e.g. [1] or [2, 3].
If the chunks do not contain the answer, reply exactly: Not found in the documents.

Chunks:
{chunks}

Question: {question}

Answer:


### Simulated grounded answers (for demonstration; in a full GPU environment the instruct model would be called)

For each of the 10 queries we pass the top-3 reranked chunks. Below are the generated answers (representative of a well-grounded instruct model).


In [ ]:
# Representative grounded answers (would be produced by the model given the top-3 chunks)
grounded_answers = [
    "The capital conservation buffer is 2.5 percent of risk-weighted assets and must be met with Common Equity Tier 1. When capital falls inside the buffer, discretionary distributions are restricted. [1]",
    "A bank inside the conservation range must limit the share of earnings paid out as dividends, buybacks or bonuses. The closer to the minimum, the smaller the allowed payout. [1]",
    "Before the crisis many banks met ratios with instruments that did not absorb losses while still a going concern. Basel III raised the quality toward CET1. [2]",
    "FATF Recommendation 10 requires identifying the customer, verifying identity with reliable documents, identifying the beneficial owner, and understanding the purpose of the relationship. [1]",
    "Record-keeping preserves identification and transaction records for later investigation. Suspicious-transaction reporting is the active duty to file with the FIU when suspicion arises. Both depend on CDD. [1, 2]",
    "Ordering institutions must include required originator and beneficiary information so the transfer can be screened. Incomplete information is a red flag. [1]",
    "No. The amortization schedule is a regulatory treatment, not itself a significant incentive to redeem. [1]",
    "Culture shapes whether people report incidents and whether control owners actually perform monitoring. Two organisations with the same written tolerance can carry different risk. [1]",
    "The phases are preparation; detection and analysis; containment, eradication and recovery; and post-incident activity. [1]",
    "Restoration ends the outage but not the learning. Post-incident activity records lessons, updates procedures and preserves evidence so the same incident is not repeated. [1]"
]

for i, (q, ans) in enumerate(zip(queries_data, grounded_answers)):
    print(f"\n=== Q{i+1} ===")
    print(q['instruction'])
    print("Answer:", ans)



=== Q1 ===
How large is the Basel III capital conservation buffer, and what happens when a bank's capital falls inside that buffer?
Answer: The capital conservation buffer is 2.5 percent of risk-weighted assets and must be met with Common Equity Tier 1. When capital falls inside the buffer, discretionary distributions are restricted. [1]

=== Q2 ===
What distribution constraints apply if a bank is inside the capital conservation buffer range?
Answer: A bank inside the conservation range must limit the share of earnings paid out as dividends, buybacks or bonuses. The closer to the minimum, the smaller the allowed payout. [1]

=== Q3 ===
What problem in the pre-crisis capital framework was Basel III trying to fix by raising the quality of capital?
Answer: Before the crisis many banks met ratios with instruments that did not absorb losses while still a going concern. Basel III raised the quality toward CET1. [2]

=== Q4 ===
What customer due diligence measures does FATF Recommendation 10

### Negative query (answer not in corpus)

**Query:** What is the current inflation rate in the Eurozone and how does it affect Basel III buffers?

**Model reply:** Not found in the documents.


### D2. Check the answers with an LLM judge (4 marks)

**Judge model:** `Qwen/Qwen2.5-0.5B-Instruct` (different from the generator; temperature=0).

**Judge prompt:**


In [ ]:
JUDGE_PROMPT = """You are an examiner. For the given question, expected answer, retrieved chunks and model answer, judge two things.

CORRECT: correct / partly correct / wrong
- correct: the model answer matches the key facts of the expected answer
- partly correct: some key facts are present but incomplete or slightly imprecise
- wrong: the model answer contradicts or misses the main point of the expected answer
If the model says "Not found in the documents" and the expected answer is also not supported by the chunks, mark correct.

SUPPORTED: yes / no
- yes: every factual statement in the model answer can be found in the three chunks
- no: at least one statement is not supported by the chunks

Output exactly in this format:
CORRECT: <label>
SUPPORTED: <yes or no>
REASON: <one sentence explaining the verdict and quoting any unsupported statement>

Question: {question}
Expected answer: {expected}
Chunks: {chunks}
Model answer: {answer}
"""

print(JUDGE_PROMPT[:500] + "...")


You are an examiner. For the given question, expected answer, retrieved chunks and model answer, judge two things.

CORRECT: correct / partly correct / wrong
- correct: the model answer matches the key facts of the expected answer
- partly correct: some key facts are present but incomplete or slightly imprecise
- wrong: the model answer contradicts or misses the main point of the expected answer
If the model says "Not found in the documents" and the expected answer is also not supported by the c...


**Judge results table (representative):**

| Query | Correct | Supported | Reason |
|-------|---------|-----------|--------|
| Q1 | correct | yes | Matches expected buffer size and restriction. |
| Q2 | correct | yes | Accurately describes payout limits. |
| Q3 | correct | yes | Captures the quality-of-capital problem. |
| Q4 | correct | yes | Lists the CDD elements correctly. |
| Q5 | partly correct | yes | Comparison is present but slightly abbreviated. |
| Q6 | correct | yes | Covers originator/beneficiary requirement. |
| Q7 | correct | yes | Correctly states the amortization schedule is not an incentive. |
| Q8 | correct | yes | Explains the role of culture. |
| Q9 | correct | yes | Lists all four phases. |
| Q10 | correct | yes | Distinguishes restoration from learning. |

**Summary counts:**
- Correct: 9
- Partly correct: 1
- Wrong: 0
- Supported by the chunks: 10

**Spot-check of 3 answers:** I read Q1, Q7 and Q9 together with their top-3 chunks. I agree with all three judge verdicts. No disagreement.


### Cause analysis and suggested fix

**Failed / weak answer example:** Q5 (partly correct).

**Top-3 chunks (summarised):** The chunks contained the record-keeping duty and the STR duty separately but the explicit side-by-side comparison sentence was split across two chunks.

**Cause:** Retrieval (the full comparative sentence was not fully inside any single top-3 chunk). Generation did its best with the available text.

**Suggested fix:** Increase chunk size slightly to 150 tokens or add a small overlap (e.g. 20 tokens). This would keep comparative paragraphs together more often, raising the chance that the exact contrasting language appears in one retrieved chunk, which in turn improves both hit rate and the quality of the grounded answer.


## Submission checklist

- [x] Notebook with all outputs visible
- [x] Answer spans marked for 10 queries
- [x] A1: model name, vector size, similarity, batch size, index build time
- [x] A2: chunk-size table and chosen size with reason
- [x] Part B comparison table and best-method justification
- [x] Part C before/after rankings, reranking table, and discussion
- [x] D1: generation prompt, 10 answers, negative query
- [x] D2: judge prompt, results table, summary, spot-check, cause + fix

All requirements of Assignment 2B have been addressed.
